In [1]:
import pandas as pd
import numpy as np
import re, pickle, gzip, os

def convert_sample_names(x):
    x = re.sub(r'^noncanonical_', '', x)
    x = re.sub(r'^canonical_', '', x)
    x = re.sub(r'_fdp$', '', x)
    return x

In [2]:
cutoffs_canon_trembl = pd.read_csv(
    './IonbotData_ionbot_ranks_300925/results/peptide_level/all_bysample_stats.csv',
    usecols=['sample','subset','dataset','database','search_type','approach',"Max rank 1% paired FDP"]
    )
cutoffs_canon_trembl = cutoffs_canon_trembl[cutoffs_canon_trembl.database!='open'].copy()
cutoffs_canon_trembl['sample'] = cutoffs_canon_trembl['sample'].apply(convert_sample_names)
cutoffs_canon_trembl['subset'] = 'Canonical'
cutoffs_canon_trembl.fillna(-1, inplace=True)
cutoffs_canon_trembl.sort_values('Max rank 1% paired FDP')

,Max rank 1% paired FDP,sample,subset,dataset,database,search_type,approach
10,1453.0,130327_o2_05_hu_C3_2hr,Canonical,PXD002057,trembl,ClosedSearch,classic
2,1574.0,130327_o2_03_hu_C2_2hr,Canonical,PXD002057,canon,ClosedSearch,classic
4,1613.0,130327_o2_05_hu_C3_2hr,Canonical,PXD002057,canon,ClosedSearch,classic
8,1622.0,130327_o2_03_hu_C2_2hr,Canonical,PXD002057,trembl,ClosedSearch,classic
6,2029.0,130327_o2_01_hu_C1_2hr,Canonical,PXD002057,trembl,ClosedSearch,classic
...,...,...,...,...,...,...,...
91,19944.0,Sample-MCF,Canonical,PXD014258,canon,OpenSearch,classic
18,20564.0,Sample-BT474,Canonical,PXD014258,canon,ClosedSearch,classic
90,20694.0,Sample-BT474,Canonical,PXD014258,canon,OpenSearch,classic
21,20707.0,Sample-BT474,Canonical,PXD014258,trembl,ClosedSearch,classic


In [3]:
cutoffs_openprot = pd.read_csv(
    './IonbotData_ionbot_ranks_300925/results/all_bysample_stats_subsets.csv',
    usecols=['sample','subset','dataset','database','search_type','approach',"Max rank 1% paired FDP"]
    )
cutoffs_openprot['sample'] = cutoffs_openprot['sample'].apply(convert_sample_names)
cutoffs_openprot.database = cutoffs_openprot.database.map({'open':'openprot'})
cutoffs_openprot.subset = cutoffs_openprot.subset.map({'noncanonical':'NonCanonical', 'canonical':'Canonical'})
cutoffs_openprot.sort_values('Max rank 1% paired FDP')

,Max rank 1% paired FDP,sample,subset,dataset,database,search_type,approach
7,14.0,130327_o2_02_hu_P1_2hr,NonCanonical,PXD002057,openprot,ClosedSearch,classic
11,15.0,130327_o2_06_hu_P3_2hr,NonCanonical,PXD002057,openprot,ClosedSearch,classic
43,15.0,AM17,NonCanonical,PXD005833,openprot,ClosedSearch,classic
45,15.0,AM19,NonCanonical,PXD005833,openprot,ClosedSearch,classic
44,16.0,AM18,NonCanonical,PXD005833,openprot,ClosedSearch,classic
...,...,...,...,...,...,...,...
36,NaN,AM10,NonCanonical,PXD005833,openprot,ClosedSearch,classic
37,NaN,AM11,NonCanonical,PXD005833,openprot,ClosedSearch,classic
39,NaN,AM13,NonCanonical,PXD005833,openprot,ClosedSearch,classic
56,NaN,130327_o2_03_hu_C2_2hr,NonCanonical,PXD002057,openprot,OpenSearch,classic


In [4]:
cutoffs = pd.concat([cutoffs_canon_trembl,cutoffs_openprot], ignore_index=True)
cutoffs.fillna(-1, inplace=True)
cutoffs['sample2'] = cutoffs.apply(
    lambda row: row['sample']+'-closed' if row['search_type']=='ClosedSearch' else row['sample'],
    axis=1
)
cutoffs[['subset','dataset','database','search_type']].value_counts(sort=False)

subset        dataset    database  search_type 
Canonical     PXD002057  canon     ClosedSearch     6
                                   OpenSearch       6
                         openprot  ClosedSearch     6
                                   OpenSearch       6
                         trembl    ClosedSearch     6
                                   OpenSearch       6
              PXD005833  canon     ClosedSearch    15
                                   OpenSearch      15
                         openprot  ClosedSearch    15
                                   OpenSearch      15
                         trembl    ClosedSearch    15
                                   OpenSearch      15
              PXD014258  canon     ClosedSearch     3
                                   OpenSearch       3
                         openprot  ClosedSearch     3
                                   OpenSearch       3
                         trembl    ClosedSearch     3
                                  

In [5]:
cutoffs2 = cutoffs.set_index(['sample2','subset','database']).to_dict()['Max rank 1% paired FDP']
cutoffs2

{('130327_o2_01_hu_C1_2hr-closed', 'Canonical', 'canon'): 2123.0,
 ('130327_o2_02_hu_P1_2hr-closed', 'Canonical', 'canon'): 4659.0,
 ('130327_o2_03_hu_C2_2hr-closed', 'Canonical', 'canon'): 1574.0,
 ('130327_o2_04_hu_P2_2hr-closed', 'Canonical', 'canon'): 3799.0,
 ('130327_o2_05_hu_C3_2hr-closed', 'Canonical', 'canon'): 1613.0,
 ('130327_o2_06_hu_P3_2hr-closed', 'Canonical', 'canon'): 3994.0,
 ('130327_o2_01_hu_C1_2hr-closed', 'Canonical', 'trembl'): 2029.0,
 ('130327_o2_02_hu_P1_2hr-closed', 'Canonical', 'trembl'): 4620.0,
 ('130327_o2_03_hu_C2_2hr-closed', 'Canonical', 'trembl'): 1622.0,
 ('130327_o2_04_hu_P2_2hr-closed', 'Canonical', 'trembl'): 3815.0,
 ('130327_o2_05_hu_C3_2hr-closed', 'Canonical', 'trembl'): 1453.0,
 ('130327_o2_06_hu_P3_2hr-closed', 'Canonical', 'trembl'): 3890.0,
 ('Sample-BT474-closed', 'Canonical', 'canon'): 20564.0,
 ('Sample-MCF-closed', 'Canonical', 'canon'): 18829.0,
 ('SampleHela-closed', 'Canonical', 'canon'): 16214.0,
 ('Sample-BT474-closed', 'Canonical

In [6]:
with gzip.open('custom-rank-cutoffs.gz','wb') as outfile:
    pickle.dump(cutoffs2, outfile)

In [7]:
# to read the pickle
with gzip.open('custom-rank-cutoffs.gz','rb') as infile:
    b = pickle.load(infile)

In [8]:
cutoffs2 == b

True